# Fase 3: Preprocesamiento y Feature Engineering Clínico

> **Proyecto:** Modelado predictivo y análisis multifactorial de riesgo de enfermedades cardíacas  
> **Objetivo:** Construir canalizaciones robustas de transformación (`ColumnTransformer` y `Pipeline`), generar nuevas variables fisiológicas/interactivas sin fuga de datos, y preparar las matrices de características para los modelos.

---

## Contenido de este Notebook Guiado
1. **Prevención de Fuga de Datos (*Data Leakage*)**: Por qué ajustar transformadores exclusivamente en Train.
2. **Ingeniería de Características Cardiovasculares** (`src/features.py`).
3. **Canalizaciones de Preprocesamiento** (`src/preprocessing.py`):
   - One-Hot Encoding para nominales (`region`, `sex`, etc.) con `drop='first'`.
   - Ordinal Encoding con orden explícito para `income_level` y `physical_activity`.
   - Escalado estándar para modelos lineales (`StandardScaler`).
4. **Ejercicios Guiados para el Alumno (TODOs)**


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    TARGET_COL,
    NOMINAL_COLS,
    ORDINAL_COLS,
    ORDINAL_CATEGORIES,
    BINARY_NUMERIC_COLS,
    CONTINUOUS_NUMERIC_COLS,
)
from src.data_loader import load_processed_splits
from src.features import CardiovascularFeatureEngineer
from src.preprocessing import build_linear_preprocessor, build_tree_preprocessor

train_df, val_df, test_df = load_processed_splits()
X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df.drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]

X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")


## 1. Implementación de Feature Engineering en `src/features.py`
En `src/features.py` se estructuró la clase `CardiovascularFeatureEngineer`.
Tu tarea en esta sección es completar las fórmulas clave descritas en los comentarios del módulo:

1. **`sleep_deviation`**: Distancia absoluta respecto al sueño óptimo (7.5 horas).
   $$ \text{sleep\_deviation} = |\text{sleep\_hours} - 7.5| $$
2. **`hr_reserve_ratio`**: Relación entre frecuencia cardíaca máxima y frecuencia cardíaca basal.
   $$ \text{hr\_reserve\_ratio} = \frac{\text{max\_heart\_rate}}{\text{avg\_resting\_hr} + 10^{-5}} $$
3. **`age_x_hypertension`**: Interacción multiplicativa entre edad e hipertensión.
   $$ \text{age\_x\_hypertension} = \text{age} \times \text{hypertension} $$
4. **`bmi_x_diabetes`**: Interacción multiplicativa metabólica.
   $$ \text{bmi\_x\_diabetes} = \text{bmi} \times \text{diabetes} $$


### 🛠️ Ejercicio 2.1: Probar y Verificar tu Feature Engineering
Una vez implementadas las fórmulas en [src/features.py](file:///home/fran/Documents/ML/src/features.py), ejecuta la siguiente celda para validar que se generen las columnas adicionales en `X_train`:


In [ ]:
# TODO: Instancia CardiovascularFeatureEngineer y transforma X_train
feature_engineer = CardiovascularFeatureEngineer()
X_train_enriched = feature_engineer.transform(X_train)

print(f"Dimensiones originales: {X_train.shape}")
print(f"Dimensiones enriquecidas: {X_train_enriched.shape}")
new_cols = [c for c in X_train_enriched.columns if c not in X_train.columns]
print(f"Nuevas variables generadas ({len(new_cols)}): {new_cols}")
display(X_train_enriched[new_cols].head())


## 2. Inspección del `ColumnTransformer`
El preprocesador para modelos lineales debe transformar las variables heterogéneas en una matriz numérica continua y acotada.


In [ ]:
linear_preprocessor = build_linear_preprocessor()

# Ajustar exclusivamente en Train para evitar Data Leakage
linear_preprocessor.fit(X_train)

X_train_proc = linear_preprocessor.transform(X_train)
X_val_proc = linear_preprocessor.transform(X_val)

feature_names = linear_preprocessor.get_feature_names_out()
print(f"Dimensión resultante para modelos lineales: {X_train_proc.shape}")
print(f"Total de características resultantes tras codificación: {len(feature_names)}")


### 🛠️ Ejercicio 2.2: Construir un Pipeline Completo (Feature Engineering + Preprocesamiento)
En Scikit-Learn, podemos encadenar `CardiovascularFeatureEngineer` y `ColumnTransformer` en un único `Pipeline`.

**Tu tarea:**
Crea un `Pipeline` que primero ejecute tu ingeniería de características y luego aplique el preprocesamiento correspondiente.


In [ ]:
# TODO: Importa Pipeline de sklearn.pipeline y combina ambos pasos:
from sklearn.pipeline import Pipeline

full_preprocessing_pipeline = Pipeline([
    ('features', CardiovascularFeatureEngineer()),
    ('preprocessor', build_linear_preprocessor())
])

X_train_full = full_preprocessing_pipeline.fit_transform(X_train)
X_val_full = full_preprocessing_pipeline.transform(X_val)

print(f"Matriz de Train transformada: {X_train_full.shape}")
print(f"Matriz de Val transformada: {X_val_full.shape}")
print(f"Número de características finales: {X_train_full.shape[1]}")
